In [0]:
from pyspark.sql import functions as F

fhv = spark.table("silver.fhv_taxi_trips").alias("fhv")
yellow = spark.table("silver.yellow_taxi_trips").alias("y")
green = spark.table("silver.green_taxi_trips").alias("g")
hvfhv = spark.table("silver.high_volume_fhv").alias("h")

In [0]:
#om vi vill droppa en kolumn är det lätt här, denna schema bestämmer vad vi ska behålla i final table
target_schema = {
    "taxi_type": "string",
    "service_type": "string",
    "Year": "int",
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "trip_time": "double",
    "dolocationid": "int",
    "pulocationid": "int",
    "trip_distance": "double",
    "extra": "decimal(10,2)",
    "fare_amount": "decimal(10,2)",
    "mta_tax": "decimal(10,2)",
    "tip_amount": "decimal(10,2)",
    "tolls_amount": "decimal(10,2)",
    "total_amount": "decimal(10,2)",
    "payment_types": "string",
    "rate_codes": "string",
    "vendors": "string",
    "base_passenger_fare": "decimal(10,2)",
    "bcf": "decimal(10,2)",
    "driver_pay": "decimal(10,2)",
    "on_scene_datetime": "timestamp",
    "request_datetime": "timestamp",
}

#funktion för att göra de olika tables matchande
def align(df):
    return df.select(*[
        (F.col(c).cast(t) if c in df.columns else F.lit(None).cast(t)).alias(c)
        for c, t in target_schema.items()
    ])

combined_df = (
    align(hvfhv)
    .unionByName(align(yellow))
    .unionByName(align(green))
    .unionByName(align(fhv))
)

In [0]:
#lägger in null istället för ogiltiga värden samt försöker beräkna olika amounts
#lägger även till monotonically increasing id (row id) för att ha en unique identifier, framförallt för geodatan
final_df = (
    combined_df
    .withColumn(
        "dolocationid",
        F.when((F.col("dolocationid") == 0) | (F.col("dolocationid") < 0) | (F.col("dolocationid") > 265), F.lit(None))
        .otherwise(F.col("dolocationid"))
    )
    .withColumn(
        "pulocationid",
        F.when((F.col("pulocationid") == 0) | (F.col("pulocationid") < 0) | (F.col("pulocationid") > 265), F.lit(None))
        .otherwise(F.col("pulocationid"))
    )
    .withColumn(
        "payment_types",
        F.when(F.col("payment_types").isin("Unknown", "Unrecognised"), F.lit(None))
        .otherwise(F.col("payment_types"))
    )
    .withColumn(
        "rate_codes",
        F.when(F.col("rate_codes").isin("Unknown", "Unrecognised", "Group ride"), F.lit(None))
        .otherwise(F.col("rate_codes"))
    )
    .withColumn(
        "vendors",
        F.when(F.col("vendors").isin("Unknown", "Unrecognised"), F.lit(None))
        .otherwise(F.col("vendors"))
    )
    .withColumn(
        "trip_time",
        F.when(F.col("trip_time") == 0, F.lit(None))
        .otherwise(F.col("trip_time"))
    )
    .withColumn(
        "trip_distance",
        F.when(F.col("trip_distance") <= 0, F.lit(None))
        .otherwise(F.col("trip_distance"))
    )



    .withColumn(
        "total_amount",
        F.when(
            (F.col("total_amount") <= 0) | (F.col("total_amount").isNull()),
            F.greatest(
                F.coalesce(F.col("base_passenger_fare"), F.lit(0)) +
                F.coalesce(F.col("bcf"), F.lit(0)) +
                F.coalesce(F.col("extra"), F.lit(0)) +
                F.coalesce(F.col("mta_tax"), F.lit(0)) +
                F.coalesce(F.col("tip_amount"), F.lit(0)) +
                F.coalesce(F.col("tolls_amount"), F.lit(0)),

                F.coalesce(F.col("fare_amount"), F.lit(0)) +
                F.coalesce(F.col("bcf"), F.lit(0)) +
                F.coalesce(F.col("extra"), F.lit(0)) +
                F.coalesce(F.col("mta_tax"), F.lit(0)) +
                F.coalesce(F.col("tip_amount"), F.lit(0)) +
                F.coalesce(F.col("tolls_amount"), F.lit(0))
            )
        ).otherwise(F.col("total_amount"))
    )
    .withColumn(
        "row_id",
        F.monotonically_increasing_id()
    )
#oklart om vi vill droppa alla som saknar loc id, många som var Unknown som nu är null, vi får kolla närmre först
#    .filter(
#        F.col("pulocationid").isNotNull() & 
#        F.col("dolocationid").isNotNull()
#    )
)

In [0]:
#find long/lat to calc distance where distance is missing
temp_df_missing_distance = final_df.select(F.col("row_id"), F.col("trip_distance"), F.col("pulocationid"), F.col("dolocationid")).where(F.col("trip_distance").isNull())

In [0]:
final_df.createOrReplaceTempView("all_taxi_trips")

In [0]:
%sql
SELECT
  Year,
  COUNT(*) AS total,

  -- NULL counts
  SUM(CASE WHEN trip_time IS NULL THEN 1 ELSE 0 END) AS trip_time_null,
  SUM(CASE WHEN trip_time = 0 THEN 1 ELSE 0 END) AS trip_time_zero,
  SUM(CASE WHEN dolocationid IS NULL THEN 1 ELSE 0 END) AS dolocationid_null,
  SUM(CASE WHEN dolocationid = 0 THEN 1 ELSE 0 END) AS dolocationid_zero,
  SUM(CASE WHEN pulocationid IS NULL THEN 1 ELSE 0 END) AS pulocationid_null,
  SUM(CASE WHEN pulocationid = 0 THEN 1 ELSE 0 END) AS pulocationid_zero,
  SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS trip_distance_null,
  SUM(CASE WHEN trip_distance = 0 THEN 1 ELSE 0 END) AS trip_distance_zero,
  SUM(CASE WHEN extra IS NULL THEN 1 ELSE 0 END) AS extra_null,
  SUM(CASE WHEN extra = 0 THEN 1 ELSE 0 END) AS extra_zero,
  SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) AS fare_amount_null,
  SUM(CASE WHEN fare_amount = 0 THEN 1 ELSE 0 END) AS fare_amount_zero,
  SUM(CASE WHEN mta_tax IS NULL THEN 1 ELSE 0 END) AS mta_tax_null,
  SUM(CASE WHEN mta_tax = 0 THEN 1 ELSE 0 END) AS mta_tax_zero,
  SUM(CASE WHEN tip_amount IS NULL THEN 1 ELSE 0 END) AS tip_amount_null,
  SUM(CASE WHEN tip_amount = 0 THEN 1 ELSE 0 END) AS tip_amount_zero,
  SUM(CASE WHEN tolls_amount IS NULL THEN 1 ELSE 0 END) AS tolls_amount_null,
  SUM(CASE WHEN tolls_amount = 0 THEN 1 ELSE 0 END) AS tolls_amount_zero,
  SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS total_amount_null,
  SUM(CASE WHEN total_amount = 0 THEN 1 ELSE 0 END) AS total_amount_zero,
  SUM(CASE WHEN base_passenger_fare IS NULL THEN 1 ELSE 0 END) AS base_passenger_fare_null,
  SUM(CASE WHEN base_passenger_fare = 0 THEN 1 ELSE 0 END) AS base_passenger_fare_zero,
  SUM(CASE WHEN bcf IS NULL THEN 1 ELSE 0 END) AS bcf_null,
  SUM(CASE WHEN bcf = 0 THEN 1 ELSE 0 END) AS bcf_zero,
  SUM(CASE WHEN driver_pay IS NULL THEN 1 ELSE 0 END) AS driver_pay_null,
  SUM(CASE WHEN driver_pay = 0 THEN 1 ELSE 0 END) AS driver_pay_zero

FROM all_taxi_trips
GROUP BY Year
ORDER BY Year;

In [0]:
%sql
SELECT payment_types, COUNT(*) AS cnt
FROM all_taxi_trips
GROUP BY payment_types
ORDER BY cnt DESC;

In [0]:
%sql
SELECT rate_codes, COUNT(*) AS cnt
FROM all_taxi_trips
GROUP BY rate_codes
ORDER BY cnt DESC;

In [0]:
%sql
SELECT vendors, COUNT(*) AS cnt
FROM all_taxi_trips
GROUP BY vendors
ORDER BY cnt DESC;